## Notebook para análise da base processada

In [2]:
## Imports

import sys
from pathlib import Path

## ============================================
# Garante a leitura do modulo src
project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root))
## ============================================

import pandas as pd
from src.data.reader import fetch_dataset
from src.data.process import clean_dataset, MISSING_VALUE

C:\Users\cadu2\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Leitura e processamento dos dados

raw_data_dir = fetch_dataset()
csv_path = next(raw_data_dir.glob("*.csv"))
raw_df = pd.read_csv(csv_path)

df = clean_dataset(raw_df)
df.head()

,title,text,date,category,subcategory,link
0,"lula diz que está 'lascado', mas que ainda tem...",com a possibilidade de uma condenação impedir ...,2017-09-10,poder,N/D,http://www1.folha.uol.com.br/poder/2017/10/192...
1,"'decidi ser escrava das mulheres que sofrem', ...","para oumou sangaré, cantora e ativista malines...",2017-09-10,ilustrada,N/D,http://www1.folha.uol.com.br/ilustrada/2017/10...
2,três reportagens da folha ganham prêmio petrob...,três reportagens da folha foram vencedoras do ...,2017-09-10,poder,N/D,http://www1.folha.uol.com.br/poder/2017/10/192...
3,filme 'star wars: os últimos jedi' ganha trail...,a disney divulgou na noite desta segunda-feira...,2017-09-10,ilustrada,N/D,http://www1.folha.uol.com.br/ilustrada/2017/10...
4,cbss inicia acordos com fintechs e quer 30% do...,"o cbss, banco da holding elopar dos sócios bra...",2017-09-10,mercado,N/D,http://www1.folha.uol.com.br/mercado/2017/10/1...


In [4]:
(df == MISSING_VALUE).sum()

title               0
text               41
date                0
category            0
subcategory    136601
link                0
dtype: int64

In [5]:
df["category"].value_counts().sort_values()

category
2015                                1
bichos                              1
2016                                1
musica                              1
contas-de-casa                      2
ombudsman                           3
euronews                            8
mulher                             15
treinamentocienciaesaude           18
treinamento                        21
multimidia                         26
guia-de-livros-filmes-discos       28
rfi                                29
especial                           43
cenarios-2017                      43
infograficos                       43
dw                                 48
banco-de-dados                     64
topofmind                          85
guia-de-livros-discos-filmes      143
vice                              146
o-melhor-de-sao-paulo             189
serafina                          334
seminariosfolha                   379
ambiente                          490
asmais                            548
com

In [6]:
tabela_nulos_por_categoria = df.groupby("category").agg(**{
    "TOTAL": ("category", "size"),
    "N/A-TEXT": ("text", lambda s: (s == MISSING_VALUE).sum()),
    "N/A-SUBCATEGORY": ("subcategory", lambda s: (s == MISSING_VALUE).sum()),
})
tabela_nulos_por_categoria.sort_values("TOTAL", ascending=False)

,TOTAL,N/A-TEXT,N/A-SUBCATEGORY
category,,,
poder,22008,0,21069
colunas,21509,3,0
mercado,20947,0,20947
esporte,19725,0,16866
mundo,17126,0,17126
cotidiano,16956,0,16921
ilustrada,15612,4,15612
opiniao,4523,0,4523
paineldoleitor,4008,0,3748


In [17]:
# Subcategoria -> categorias distintas em que ela aparece
df.groupby("subcategory")["category"].agg(**{ 
                                        "categorias": lambda s: ", ".join(sorted(s.unique())),
                                        "num_categorias": lambda s: len(s.unique()),
                                        "total_ocorrencias": lambda s: len(s),
                                    })\
                                    .reset_index('subcategory')\
                                    .sort_values("num_categorias", ascending=False)\
                                    .head(20)

,subcategory,categorias,num_categorias,total_ocorrencias
0,N/D,"2015, 2016, ambiente, asmais, banco-de-dados, ...",43,136601
1,acidadeesua,paineldoleitor,1,194
2,adrianagomes,colunas,1,39
3,aecioneves,colunas,1,115
4,agendafolha,paineldoleitor,1,51
5,alessandra-orofino,colunas,1,16
6,alexandracorvo,colunas,1,44
7,alexandraforbes,colunas,1,65
8,alexandreschwartsman,colunas,1,140
9,alexandrevidalporto,colunas,1,72


In [9]:
# Categoria -> subcategorias distintas em que ela aparece
df[df["subcategory"] != MISSING_VALUE].groupby("category")["subcategory"]\
                                        .agg(
                                            subcategorias=lambda s: ", ".join(sorted(s.unique())),
                                            num_subcategorias=lambda s: s.nunique(),
                                        ).reset_index()\
                                        .sort_values("num_subcategorias", ascending=False)\
                                        .head(20)

,category,subcategorias,num_subcategorias
0,colunas,"adrianagomes, aecioneves, alessandra-orofino, ...",233
11,tv,"ambiente, arena-do-marketing, cebrap, ciencia,...",37
10,sobretudo,"carreiras, loja, morar, natural, negocios, rod...",7
7,paineldoleitor,"acidadeesua, agendafolha, meuolhar, semanadole...",4
3,empreendedorsocial,"colunas, entrelinhassociais, minhahistoria",3
6,o-melhor-de-sao-paulo,"noivas-e-casamentos, servicos",2
9,saopaulo,"hoje, viaja-sp",2
1,contas-de-casa,seu-bolso,1
2,cotidiano,ribeiraopreto,1
5,multimidia,tvfolha,1
